# Calcular rasgos derivados — `Rasgos_2025` (`Sumaco_17092025.xlsx`)

Calcula, a partir de las columnas ya medidas en campo:
- **Wood density** (g/cm³)
- **Wood specific gravity (WSG)** (adimensional)
- **Stem water content** (%)
- **Mean leaf thickness** (mm)
- **Dry matter content / LDMC** (mg/g)

⚠️ **`Force to punch mean (kN/m)` no se pudo calcular**: revisé las 4 hojas del archivo (`Rasgos_2025`, `final`, `plots`, `comm`) y no hay ninguna columna con la fuerza cruda del punzón (penetrómetro) — solo están los datos de peso/grosor de hoja y de madera. Para calcular ese rasgo necesitas la fuerza medida (N) y normalmente el perímetro/diámetro del punzón; si esos datos existen en otro archivo, dime y lo incorporo.

## 1. Cargar los datos

In [18]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_excel('Sumaco_17092025.xlsx', sheet_name='Rasgos_2025')
print(df.shape)

(45, 40)


## 2. Wood density y WSG

El barrenador (increment borer) usado tiene **0.5 cm de diámetro** → radio = 0.25 cm. El volumen del cilindro de madera extraído se calcula con la longitud húmeda (`wet lenght cm`):

$$V\ (cm^3) = \pi \times r^2 \times L$$

- **Wood density (g/cm³)** = peso seco de madera / V
- **WSG** = wood density / densidad del agua (1 g/cm³) → numéricamente el mismo valor, pero adimensional (es el estándar en ecología de bosques, ej. Chave et al. 2006).

In [19]:
radio_barrenador_cm = 0.5 / 2  # 0.25 cm

df['wood_volume_cm3'] = np.pi * radio_barrenador_cm**2 * df['wet lenght cm']
df['wood_density_g_cm3'] = df['dry wood weight'] / df['wood_volume_cm3']
df['WSG'] = df['wood_density_g_cm3'] / 1.0  # densidad del agua = 1 g/cm3

df[['wood_volume_cm3', 'wood_density_g_cm3', 'WSG']].describe().T

,count,mean,std,min,25%,50%,75%,max
wood_volume_cm3,17.0,1.220832,0.418841,0.431969,0.883573,1.256637,1.551161,1.865321
wood_density_g_cm3,17.0,0.502524,0.106908,0.286479,0.497721,0.519906,0.558132,0.696038
WSG,17.0,0.502524,0.106908,0.286479,0.497721,0.519906,0.558132,0.696038


## 3. Stem water content (%)

Contenido de agua del tallo, como porcentaje del peso seco:

$$SWC\ (\%) = \frac{peso\ humedo - peso\ seco}{peso\ seco} \times 100$$

In [20]:
df['stem_water_content_pct'] = (
    (df['wet wood weight'] - df['dry wood weight']) / df['dry wood weight'] * 100
)
df['stem_water_content_pct'].describe()

count     17.000000
mean     118.296717
std       36.676439
min       73.170732
25%       88.372093
50%      113.043478
75%      132.894737
max      206.349206
Name: stem_water_content_pct, dtype: float64

## 4. Mean leaf thickness (mm)

Promedio de las 6 mediciones de grosor (3 hojas x 2 puntos cada una).

In [21]:
grosor_cols = ['leaf 1 thickness 1', 'leaf 1 thickness 2',
               'leaf 2 thickness 1', 'leaf 2 thickness 2',
               'leaf 3 thickness 1', 'leaf 3 thickness 2']

df['mean_leaf_thickness_mm'] = df[grosor_cols].mean(axis=1)
df['mean_leaf_thickness_mm'].describe()

count    44.000000
mean      0.233261
std       0.062990
min       0.138333
25%       0.181917
50%       0.233250
75%       0.260417
max       0.439500
Name: mean_leaf_thickness_mm, dtype: float64

## 5. Dry matter content / LDMC (mg/g)

Contenido de materia seca de la hoja: masa seca (en mg) dividida entre la masa fresca (en g).

$$LDMC\ (mg/g) = \frac{peso\ seco\ hoja\ (g) \times 1000}{peso\ fresco\ hoja\ (g)}$$

In [22]:
df['LDMC_mg_g'] = df['Leaf dry weight (g)'] * 1000 / df['Leaf fresh weight (g)']
df['LDMC_mg_g'].describe()

count     45.000000
mean     326.114169
std       93.828272
min      183.030303
25%      242.955326
50%      309.502425
75%      407.769488
max      524.156703
Name: LDMC_mg_g, dtype: float64

# 5. 

In [23]:
LA = pd.read_excel("Sumaco_LA.xlsx")

df = pd.merge(df,LA, how="left",right_on="new_TreeID",left_on="new tree ID 2025")

## 6. Revisar rangos y valores fuera de lo esperado

Rangos de referencia típicos para bosques tropicales (ajustables si tu sistema usa otros umbrales):
- Wood density / WSG: 0.15 – 1.2 g/cm³
- Stem water content: 30 – 250 %
- Mean leaf thickness: 0.05 – 1.0 mm
- LDMC: 100 – 600 mg/g

In [24]:
rangos = {
    'wood_density_g_cm3': (0.15, 1.2),
    'WSG': (0.15, 1.2),
    'stem_water_content_pct': (30, 250),
    'mean_leaf_thickness_mm': (0.05, 1.0),
    'LDMC_mg_g': (100, 600),
}

df['QC_flag_rasgos'] = ''
for col, (lo, hi) in rangos.items():
    fuera = df[col].notna() & ((df[col] < lo) | (df[col] > hi))
    df.loc[fuera, 'QC_flag_rasgos'] += f'{col} fuera de rango [{lo}-{hi}]; '

print('Filas fuera de rango:', (df['QC_flag_rasgos'] != '').sum(), 'de', len(df))
df[df['QC_flag_rasgos'] != ''][['plot ID', 'treeID', 'genus', 'species', 'QC_flag_rasgos']]

Filas fuera de rango: 0 de 45


,plot ID,treeID,genus,species,QC_flag_rasgos


## 7. Guardar resultado

In [25]:
with pd.ExcelWriter('Rasgos_2025_con_rasgos_calculados.xlsx', engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Rasgos_2025_calculado', index=False)

print('Guardado: Rasgos_2025_con_rasgos_calculados.xlsx')

Guardado: Rasgos_2025_con_rasgos_calculados.xlsx
